In [68]:
import numpy as np
import time
from pynq import Overlay, allocate

# 1. Load the overlay
ol = Overlay("dac_adc.bit")

In [69]:
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel

In [80]:
import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate
from scipy.signal import windows

# --- INITIALIZATION ---
ol = Overlay("dac_adc.bit")
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Hardware Constants
ADC_SAMPLE_RATE_MSPS = 1638.4 
DATA_SIZE_RECV = 8192 
TARGET_FREQS = [100, 200, 300, 400]
# Mapping frequency back to bits
FREQ_TO_BITS = {100: "00", 200: "01", 300: "10", 400: "11"}

# Allocate Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(DATA_SIZE_RECV,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def capture_wave():
    dma_recv.transfer(output_buffer)
    dma_recv.wait()
    return output_buffer

def analyze_current_signal(data):
    """Your specific working ADC logic"""
    centered_data = data.astype(np.float32) - np.mean(data)
    win = windows.blackmanharris(len(centered_data))
    windowed_signal = centered_data * win
    
    n = len(windowed_signal)
    fft_res = np.fft.fft(windowed_signal)
    freqs = np.fft.fftfreq(n, 1/(ADC_SAMPLE_RATE_MSPS * 1e6))
    
    ignore_bins = 20
    pos_freqs = freqs[ignore_bins:n//2] / 1e6
    pos_mag = np.abs(fft_res[ignore_bins:n//2])
    
    # Return the frequency of the highest peak
    peak_freq = pos_freqs[np.argmax(pos_mag)]
    return peak_freq

def play_and_analyze_sequence(sequence):
    print(f"Starting Sequence: {sequence}")
    decoded_string = ""
    
    # 1. Start the 'Digital DC' carrier (DAC side)
    input_buffer[:] = 0x7FFF 
    dma_send.transfer(input_buffer) 
    
    # 2. Loop through the sequence
    for i in range(0, len(sequence), 2):
        pair = sequence[i:i+2]
        
        if pair == "00": freq = 100
        elif pair == "01": freq = 200
        elif pair == "10": freq = 300
        elif pair == "11": freq = 400
        else: continue
            
        # Update DAC Frequency
        update_nco(target_block, freq) 
      
        
        # --- Capture and Decode ---
        raw_adc_data = capture_wave()
        detected_f = analyze_current_signal(raw_adc_data)
        
        # Find the closest matching target frequency to decode bits
        closest_freq = min(TARGET_FREQS, key=lambda x: abs(x - detected_f))
        detected_bits = FREQ_TO_BITS[closest_freq]
        
        decoded_string += detected_bits
        print(f"TX: {pair} -> Detected: {detected_f:.2f} MHz -> RX Bits: {detected_bits}")
        
        #time.sleep(0.5)

    # 3. SHUTDOWN
    print("\nSequence complete. Stopping signal output...")
    input_buffer[:] = 0 
    dma_send.transfer(input_buffer) 
    update_nco(target_block, 0)
    
    print("-" * 30)
    print(f"Original Sequence: {sequence}")
    print(f"Decoded  Sequence: {decoded_string}")
    print("-" * 30)

# --- Run ---
try:
    user_seq = input("Enter binary sequence (e.g. 00011011): ")
    if len(user_seq) % 2 != 0:
        print("Please enter an even number of bits.")
    else:
        play_and_analyze_sequence(user_seq)
except KeyboardInterrupt:
    print("\nStopped.")

Enter binary sequence (e.g. 00011011):  00011011000110110001101100011011000110110001101100


Starting Sequence: 00011011000110110001101100011011000110110001101100
TX: 00 -> Detected: 100.80 MHz -> RX Bits: 00
TX: 01 -> Detected: 201.40 MHz -> RX Bits: 01
TX: 10 -> Detected: 302.40 MHz -> RX Bits: 10
TX: 11 -> Detected: 397.80 MHz -> RX Bits: 11
TX: 00 -> Detected: 100.80 MHz -> RX Bits: 00
TX: 01 -> Detected: 201.60 MHz -> RX Bits: 01
TX: 10 -> Detected: 302.60 MHz -> RX Bits: 10
TX: 11 -> Detected: 397.80 MHz -> RX Bits: 11
TX: 00 -> Detected: 100.80 MHz -> RX Bits: 00
TX: 01 -> Detected: 201.80 MHz -> RX Bits: 01
TX: 10 -> Detected: 302.20 MHz -> RX Bits: 10
TX: 11 -> Detected: 397.20 MHz -> RX Bits: 11
TX: 00 -> Detected: 100.80 MHz -> RX Bits: 00
TX: 01 -> Detected: 201.40 MHz -> RX Bits: 01
TX: 10 -> Detected: 302.00 MHz -> RX Bits: 10
TX: 11 -> Detected: 397.60 MHz -> RX Bits: 11
TX: 00 -> Detected: 100.80 MHz -> RX Bits: 00
TX: 01 -> Detected: 201.40 MHz -> RX Bits: 01
TX: 10 -> Detected: 302.60 MHz -> RX Bits: 10
TX: 11 -> Detected: 397.20 MHz -> RX Bits: 11
TX: 00 -> 

In [74]:
import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate,PL
from scipy.signal import windows
PL.reset()
# --- INITIALIZATION ---
ol = Overlay("dac_adc.bit")
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Hardware Constants
ADC_SAMPLE_RATE_MSPS = 1638.4 
DATA_SIZE_RECV = 8192 
TARGET_FREQS = [100, 200, 300, 400]
FREQ_TO_BITS = {100: "00", 200: "01", 300: "10", 400: "11"}

# Allocate Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(DATA_SIZE_RECV,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def analyze_current_signal(data):
    """FFT Logic to find the dominant frequency peak"""
    centered_data = data.astype(np.float32) - np.mean(data)
    win = windows.blackmanharris(len(centered_data))
    windowed_signal = centered_data * win
    
    n = len(windowed_signal)
    fft_res = np.fft.fft(windowed_signal)
    freqs = np.fft.fftfreq(n, 1/(ADC_SAMPLE_RATE_MSPS * 1e6))
    
    ignore_bins = 20
    pos_freqs = freqs[ignore_bins:n//2] / 1e6
    pos_mag = np.abs(fft_res[ignore_bins:n//2])
    
    peak_freq = pos_freqs[np.argmax(pos_mag)]
    return peak_freq

In [75]:
# 1. Start the 'Digital DC' carrier (DAC side)
input_buffer[:] = 0x7FFF 
dma_send.transfer(input_buffer) 
print("DAC is now active. Ready for sequence.")

DAC is now active. Ready for sequence.


In [78]:
user_seq = input("Enter binary sequence (e.g. 00011011): ")

if len(user_seq) % 2 != 0:
    print("Error: Please enter an even number of bits.")
else:
    num_bits = len(user_seq)
    decoded_string = ""
    
    print(f"--- Starting FSK Transmission of {num_bits} bits ---")
    
    # --- Start Timing ---
    t_start = time.perf_counter()
    
    for i in range(0, num_bits, 2):
        pair = user_seq[i:i+2]
        
        # Map bits to frequency
        if pair == "00": freq = 100
        elif pair == "01": freq = 200
        elif pair == "10": freq = 300
        elif pair == "11": freq = 400
        
        # Update DAC
        update_nco(target_block, freq) 
        
        # Capture from ADC
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        # Decode via FFT
        detected_f = analyze_current_signal(output_buffer)
        closest_freq = min(TARGET_FREQS, key=lambda x: abs(x - detected_f))
        decoded_bits = FREQ_TO_BITS[closest_freq]
        decoded_string += decoded_bits
        
        # Optional sleep (remove or set to 0 for max speed)
        # time.sleep(0.01)

    t_stop = time.perf_counter()
    # --- End Timing ---

    # Calculations
    duration = t_stop - t_start
    bps = num_bits / duration
    correct_bits = sum(1 for a, b in zip(user_seq, decoded_string) if a == b)
    accuracy = (correct_bits / num_bits) * 100

    print("\n" + "="*45)
    print(f"FSK PERFORMANCE RESULTS")
    print("="*45)
    print(f"Original:   {user_seq}")
    print(f"Decoded:    {decoded_string}")
    print(f"Accuracy:   {accuracy:.2f}%")
    print("-" * 45)
    print(f"Total Time: {duration:.4f} seconds")
    print(f"Data Rate:  {bps:.2f} bits per second (bps)")
    print("="*45)

    # Shutdown Output
    input_buffer[:] = 0 
    dma_send.transfer(input_buffer) 
    update_nco(target_block, 0)

Enter binary sequence (e.g. 00011011):  00011011000110110001101100011011000110110001101100


--- Starting FSK Transmission of 50 bits ---

FSK PERFORMANCE RESULTS
Original:   00011011000110110001101100011011000110110001101100
Decoded:    11010011111111111111001111111101110011001111110011
Accuracy:   40.00%
---------------------------------------------
Total Time: 0.9756 seconds
Data Rate:  51.25 bits per second (bps)
